# Module 1: Raw Layer DQ - Catch Issues at the Gate

## Learning Objectives
- Attach system Data Metric Functions (DMFs) to raw/staging tables
- Understand table-level vs column-level DMFs
- Set automated schedules with `TRIGGER_ON_CHANGES`
- Query DQ monitoring results
- Add basic expectations for pass/fail gates

## Key Concept: Shift-Left Data Quality

> **"Don't wait until Gold to find problems. Catch them at the gate."**

Traditional DQ approaches validate data only after expensive transformations. Snowflake's native DQ monitoring lets you attach checks directly to raw tables -- problems are detected the moment data lands.

### System DMFs Available

| DMF | Level | What It Measures |
|-----|-------|-----------------|
| `SNOWFLAKE.CORE.ROW_COUNT` | Table | Number of rows (volume check) |
| `SNOWFLAKE.CORE.FRESHNESS` | Column (TIMESTAMP) | Seconds since last DML |
| `SNOWFLAKE.CORE.NULL_COUNT` | Column | Count of NULL values |
| `SNOWFLAKE.CORE.BLANK_COUNT` | Column | Count of empty strings |
| `SNOWFLAKE.CORE.DUPLICATE_COUNT` | Column | Count of non-unique values |
| `SNOWFLAKE.CORE.UNIQUE_COUNT` | Column | Count of distinct values |

---

> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes

---
## The 7 Data Quality Domains

Every check in this lab maps to one of these domains. Each module badge tells you which domain you're working on.

| Domain | Question | Example from Our Data | Snowflake Check |
|--------|----------|----------------------|-----------------|
| **Accuracy** | Is the format correct? | National ID `98765` is too short (must be 10 digits) | Custom DMF with RLIKE regex |
| **Completeness** | Are required fields filled? | CRM has 3 NULL National IDs (not mandatory in Salesforce) | SNOWFLAKE.CORE.NULL_COUNT |
| **Uniqueness** | Are there duplicates? | Abdullah appears in both ERP and CRM (same ID, different email) | Custom DMF: COUNT(*) - COUNT(DISTINCT) |
| **Freshness** | Is it recent enough? | Transaction loaded 3 days ago, SLA is 2 hours | SNOWFLAKE.CORE.FRESHNESS |
| **Validity** | Is it an allowed value? | City must be one of: Riyadh, Jeddah, Dammam, Makkah... | ENUM rule in Rules Catalog |
| **Volume** | Did enough data arrive? | ERP feed should have 5-100 rows. Zero means pipeline failed. | SNOWFLAKE.CORE.ROW_COUNT + bounds |
| **Consistency** | Do related fields agree? | ERP customers MUST have IBAN. Gov Portal records MUST have GOV_SERVICE. | Multi-column DMF |

> **Key Insight:** A table can pass 6 domains and still be unusable if the 7th fails. Comprehensive DQ means checking ALL domains, not just the obvious ones.

---
## Setup: Use the correct role and schema

> **Requirement:** Data Metric Functions and `DATA_METRIC_SCHEDULE` require Snowflake **Enterprise Edition** or higher. If you see permission errors, confirm your edition: `SELECT CURRENT_ACCOUNT_EDITION();`

In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE SCHEMA RAW;
USE WAREHOUSE COMPUTE_WH;

---
## 1a. Volume Check

> **Business Value:** A missing feed means reports run on stale data. Finance discovers discrepancies days later, costing hours of reconciliation. Volume checks catch this in seconds.: Did the ERP feed arrive?

`ROW_COUNT` is a **table-level** DMF (no column argument). It confirms data actually landed.

> **DQ Domain:** Volume | **Severity:** HIGH

In [ ]:
ALTER TABLE CORP_DWH.RAW.STG_CUSTOMERS_ERP
    ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.ROW_COUNT ON ();

---
## 1a-2. Volume Bounds

> **Business Value:** A feed that drops from 1000 to 5 rows signals a silent pipeline failure. Without bounds checking, downstream reports look normal but are missing 99.5% of data.: Is the row count within expected range?

> **DQ Domain:** Volume | **Severity:** MEDIUM

Basic `ROW_COUNT > 0` confirms data arrived, but in production you need **bounds**. A feed that normally delivers 1000 rows but suddenly delivers 5 rows is a problem even though it's not empty.

In [ ]:
-- Custom DMF: row count must be within expected bounds
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_VOLUME_BOUNDS(
    ARG_T TABLE(ARG_C STRING)
)
RETURNS NUMBER
AS
$$
    SELECT CASE
        WHEN (SELECT COUNT(*) FROM ARG_T) < 3 THEN 1   -- below minimum
        WHEN (SELECT COUNT(*) FROM ARG_T) > 1000 THEN 1 -- above maximum
        ELSE 0
    END
$$;

-- Attach to ERP table (expected: 5-100 rows for this feed)
ALTER TABLE CORP_DWH.RAW.STG_CUSTOMERS_ERP
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_VOLUME_BOUNDS ON (CUSTOMER_NAME_EN);

---
## 1b. Freshness

> **Business Value:** A 2-hour SLA on transaction data ensures daily reconciliation reports are always based on fresh numbers -- preventing costly month-end restatements.: Is the ERP feed on time?

SLA: data must arrive within 2 hours. `FRESHNESS` on a `TIMESTAMP_LTZ` column returns seconds since last DML.

> **DQ Domain:** Freshness | **Severity:** HIGH

In [ ]:
ALTER TABLE CORP_DWH.RAW.STG_CUSTOMERS_ERP
    ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.FRESHNESS ON (LOADED_AT);

---
## 1c. Completeness

> **Business Value:** Missing National IDs block KYC compliance. Each incomplete record is a regulatory risk that could result in fines during audit.: Are mandatory fields present in CRM?

CRM is notorious for missing National IDs (not mandatory in their system). Let's catch this at ingestion.

> **DQ Domain:** Completeness | **Severity:** CRITICAL

In [ ]:
ALTER TABLE CORP_DWH.RAW.STG_CUSTOMERS_CRM
    ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.NULL_COUNT ON (NATIONAL_ID);

ALTER TABLE CORP_DWH.RAW.STG_CUSTOMERS_CRM
    ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.NULL_COUNT ON (EMAIL);

---
## Checkpoint 1: Verify DMFs Are Attached

Run the cell below to confirm your DMFs were successfully attached to the tables.

In [ ]:
-- This query shows all DMFs currently attached to our RAW tables
SELECT
    REF_ENTITY_NAME AS TABLE_NAME,
    METRIC_DATABASE || '.' || METRIC_SCHEMA || '.' || METRIC_NAME AS DMF_FULL_NAME,
    REF_ENTITY_DOMAIN,
    METRIC_NAME,
    ARGUMENTS
FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
    REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_CUSTOMERS_ERP',
    REF_ENTITY_DOMAIN => 'TABLE'
))
UNION ALL
SELECT
    REF_ENTITY_NAME,
    METRIC_DATABASE || '.' || METRIC_SCHEMA || '.' || METRIC_NAME,
    REF_ENTITY_DOMAIN,
    METRIC_NAME,
    ARGUMENTS
FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
    REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_CUSTOMERS_CRM',
    REF_ENTITY_DOMAIN => 'TABLE'
))
ORDER BY TABLE_NAME, METRIC_NAME;

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Verify DMF attachments
erp_dmfs = session.sql("""
SELECT COUNT(*) AS CNT FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
    REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_CUSTOMERS_ERP', REF_ENTITY_DOMAIN => 'TABLE'))
""").collect()[0]['CNT']

crm_dmfs = session.sql("""
SELECT COUNT(*) AS CNT FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
    REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_CUSTOMERS_CRM', REF_ENTITY_DOMAIN => 'TABLE'))
""").collect()[0]['CNT']

print("=" * 50)
print("CHECKPOINT 1: DMF Attachment Verification")
print("=" * 50)
tests_passed = 0

# Test 1: ERP should have 2 DMFs (ROW_COUNT + FRESHNESS)
if erp_dmfs >= 2:
    print(f"  [PASS] STG_CUSTOMERS_ERP has {erp_dmfs} DMFs attached (expected >= 2)")
    tests_passed += 1
else:
    print(f"  [FAIL] STG_CUSTOMERS_ERP has {erp_dmfs} DMFs (expected >= 2)")

# Test 2: CRM should have 2 DMFs (NULL_COUNT on NATIONAL_ID + EMAIL)
if crm_dmfs >= 2:
    print(f"  [PASS] STG_CUSTOMERS_CRM has {crm_dmfs} DMFs attached (expected >= 2)")
    tests_passed += 1
else:
    print(f"  [FAIL] STG_CUSTOMERS_CRM has {crm_dmfs} DMFs (expected >= 2)")

print(f"\nResult: {tests_passed}/2 checks passed")
print("=" * 50)

---
## 1d. Transactions

> **Business Value:** Blank customer references mean revenue cannot be attributed. This directly impacts commission calculations and customer profitability reporting.: Freshness + Completeness + Blanks

The transaction feed has a blank CUSTOMER_REF (empty string). `NULL_COUNT` won't catch it -- we need `BLANK_COUNT`.

> **DQ Domain:** Completeness + Accuracy | **Severity:** HIGH

In [ ]:
ALTER TABLE CORP_DWH.RAW.STG_TRANSACTIONS
    ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.FRESHNESS ON (LOADED_AT);

ALTER TABLE CORP_DWH.RAW.STG_TRANSACTIONS
    ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.NULL_COUNT ON (CUSTOMER_REF);

ALTER TABLE CORP_DWH.RAW.STG_TRANSACTIONS
    ADD DATA METRIC FUNCTION SNOWFLAKE.CORE.BLANK_COUNT ON (CUSTOMER_REF);

---
## 1e. Set Schedules: Trigger on Changes

DMFs can run on a schedule (cron) or automatically when data changes. `TRIGGER_ON_CHANGES` = zero maintenance.

> **DQ Domain:** All (infrastructure) | **Severity:** N/A

In [ ]:
ALTER TABLE CORP_DWH.RAW.STG_CUSTOMERS_ERP SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';
ALTER TABLE CORP_DWH.RAW.STG_CUSTOMERS_CRM SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';
ALTER TABLE CORP_DWH.RAW.STG_GOV_PORTAL SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';
ALTER TABLE CORP_DWH.RAW.STG_TRANSACTIONS SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';

---
## 1f. Add Expectations: Pass/Fail Gates

Expectations turn numeric metrics into actionable **verdicts**.

> **DQ Domain:** Freshness + Completeness | **Severity:** HIGH

In [ ]:
-- Freshness must be <= 2 hours (7200 seconds)
ALTER TABLE CORP_DWH.RAW.STG_TRANSACTIONS
    MODIFY DATA METRIC FUNCTION SNOWFLAKE.CORE.FRESHNESS ON (LOADED_AT)
    ADD EXPECTATION EXPECT_RAW_FEED_FRESH (VALUE <= 7200);

-- No blank customer refs
ALTER TABLE CORP_DWH.RAW.STG_TRANSACTIONS
    MODIFY DATA METRIC FUNCTION SNOWFLAKE.CORE.BLANK_COUNT ON (CUSTOMER_REF)
    ADD EXPECTATION EXPECT_NO_BLANK_REFS (VALUE = 0);

---
## 1g. View Results

Results appear after DMFs execute (1-2 minutes or immediately after DML).

In [ ]:
SELECT
    METRIC_NAME,
    ARGUMENT_NAMES,
    VALUE,
    EXPECTATION_NAME,
    EXPECTATION_RESULT,
    MEASUREMENT_TIME
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_TRANSACTIONS',
    REF_ENTITY_DOMAIN => 'TABLE'
))
ORDER BY MEASUREMENT_TIME DESC;

---
## Checkpoint 2: Verify Your Understanding

Run the verification cell below. It will check that:
1. All 4 RAW tables have schedules set
2. STG_TRANSACTIONS has 3 DMFs attached
3. Expectations are configured

---
## Drill Down: Which SPECIFIC Records Failed?

When a DMF reports "3 NULL National IDs" or "1 blank CUSTOMER_REF", the next question is always: **which records?** Here's how to find them.

In [ ]:
-- Find the 3 CRM records with NULL National IDs
SELECT FULL_NAME, NATIONAL_ID, EMAIL, MOBILE, CITY
FROM CORP_DWH.RAW.STG_CUSTOMERS_CRM
WHERE NATIONAL_ID IS NULL;

> **What this does:** Finds transaction records with blank (empty string) CUSTOMER_REF values that NULL_COUNT would miss.

In [ ]:
-- Find the blank CUSTOMER_REF in transactions (the tricky one!)
SELECT RAW_ID, CUSTOMER_REF, TXN_DATE_STR, AMOUNT_STR,
    CASE
        WHEN CUSTOMER_REF IS NULL THEN 'NULL (missing entirely)'
        WHEN CUSTOMER_REF = '' THEN 'BLANK (empty string -- looks filled but is not!)'
        ELSE 'OK'
    END AS FAILURE_REASON
FROM CORP_DWH.RAW.STG_TRANSACTIONS
WHERE CUSTOMER_REF IS NULL OR CUSTOMER_REF = '';

> **What this does:** Queries stale transaction records where LOADED_AT exceeds the 2-hour SLA threshold.

In [ ]:
-- Find stale records (LOADED_AT > 2 hours ago)
SELECT RAW_ID, CUSTOMER_REF, AMOUNT_STR, LOADED_AT,
    DATEDIFF(HOUR, LOADED_AT, CURRENT_TIMESTAMP()) AS HOURS_STALE,
    'SLA BREACH: ' || DATEDIFF(HOUR, LOADED_AT, CURRENT_TIMESTAMP()) || ' hours old (max 2)' AS FAILURE_REASON
FROM CORP_DWH.RAW.STG_TRANSACTIONS
WHERE DATEDIFF(SECOND, LOADED_AT, CURRENT_TIMESTAMP()) > 7200;

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 50)
print("CHECKPOINT 2: Full Raw Layer DQ Verification")
print("=" * 50)
tests_passed = 0
total_tests = 4

# Test 1: Transactions should have 3 DMFs
txn_dmfs = session.sql("""
SELECT COUNT(*) AS CNT FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
    REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_TRANSACTIONS', REF_ENTITY_DOMAIN => 'TABLE'))
""").collect()[0]['CNT']

if txn_dmfs >= 3:
    print(f"  [PASS] STG_TRANSACTIONS has {txn_dmfs} DMFs (expected >= 3)")
    tests_passed += 1
else:
    print(f"  [FAIL] STG_TRANSACTIONS has {txn_dmfs} DMFs (expected >= 3)")

# Test 2: Check schedule is set
schedule_check = session.sql("""
SELECT COUNT(*) AS CNT
FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
    REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_CUSTOMERS_ERP', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE SCHEDULE_STATUS = 'STARTED'
""").collect()[0]['CNT']

if schedule_check > 0:
    print(f"  [PASS] STG_CUSTOMERS_ERP schedule is active")
    tests_passed += 1
else:
    print(f"  [FAIL] STG_CUSTOMERS_ERP schedule not active (run 1e again)")

# Test 3: CRM NULL_COUNT should find 3 nulls (if results are in)
try:
    null_result = session.sql("""
    SELECT VALUE FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
        REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_CUSTOMERS_CRM', REF_ENTITY_DOMAIN => 'TABLE'))
    WHERE METRIC_NAME = 'NULL_COUNT' AND ARGUMENT_NAMES LIKE '%NATIONAL_ID%'
    ORDER BY MEASUREMENT_TIME DESC LIMIT 1
    """).collect()
    if null_result and null_result[0]['VALUE'] == 3:
        print(f"  [PASS] CRM NULL_COUNT(NATIONAL_ID) = 3 (correct!)")
        tests_passed += 1
    elif null_result:
        print(f"  [WARN] CRM NULL_COUNT(NATIONAL_ID) = {null_result[0]['VALUE']} (expected 3)")
        tests_passed += 1  # still pass, value might differ due to timing
    else:
        print(f"  [WAIT] CRM results not available yet - DMFs still running. This is normal.")
        tests_passed += 1
except:
    print(f"  [WAIT] Results not available yet - give it 1-2 minutes")
    tests_passed += 1

# Test 4: BLANK_COUNT should find 1 blank (empty string in CUSTOMER_REF)
try:
    blank_result = session.sql("""
    SELECT VALUE FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
        REF_ENTITY_NAME => 'CORP_DWH.RAW.STG_TRANSACTIONS', REF_ENTITY_DOMAIN => 'TABLE'))
    WHERE METRIC_NAME = 'BLANK_COUNT'
    ORDER BY MEASUREMENT_TIME DESC LIMIT 1
    """).collect()
    if blank_result and blank_result[0]['VALUE'] == 1:
        print(f"  [PASS] BLANK_COUNT(CUSTOMER_REF) = 1 (caught the empty string!)")
        tests_passed += 1
    elif blank_result:
        print(f"  [INFO] BLANK_COUNT = {blank_result[0]['VALUE']}")
        tests_passed += 1
    else:
        print(f"  [WAIT] BLANK_COUNT results not available yet")
        tests_passed += 1
except:
    print(f"  [WAIT] Results pending")
    tests_passed += 1

print(f"\nResult: {tests_passed}/{total_tests} checks passed")
print("=" * 50)

---
## Quiz: Test Your Knowledge

Answer these questions, then run the answer cell to check.

**Q1:** What is the difference between `NULL_COUNT` and `BLANK_COUNT`?

**Q2:** If a table's `FRESHNESS` metric returns 10800, does it pass an expectation of `VALUE <= 7200`?

**Q3:** How many NULL National IDs does the CRM feed have? (Hint: check the data loaded in Module 0)

**Q4:** Why do we use `TRIGGER_ON_CHANGES` instead of a cron schedule for raw tables?

> **What this does:** Reveals quiz answers. Try answering first!

In [ ]:
# Run this cell to reveal the answers
print("""
QUIZ ANSWERS
============

Q1: NULL_COUNT counts values that are SQL NULL (absence of value).
    BLANK_COUNT counts empty strings ('') which are NOT null but contain no data.
    Many source systems use empty strings instead of NULL -- BLANK_COUNT catches these.

Q2: NO - it FAILS. 10800 seconds = 3 hours, which exceeds the 7200-second (2-hour) threshold.
    The expectation result would be NOT_MET.

Q3: 3 NULL National IDs. The CRM feed has 6 rows total:
    - Sara Al-Dosari: NULL
    - Omar Al-Qahtani: NULL
    - Huda Al-Shehri: NULL
    - The other 3 have valid National IDs.

Q4: TRIGGER_ON_CHANGES is ideal for raw tables because:
    - Data arrives at unpredictable times (batch loads, streaming)
    - No wasted compute if no data arrives
    - Immediate detection (no waiting for next cron window)
    - Zero maintenance (no schedule to manage)
    A cron schedule is better for tables that need periodic checks regardless of changes.
""")

---
## Challenge (Self-Guided)

1. Add `DUPLICATE_COUNT` on `STG_CUSTOMERS_CRM.FULL_NAME` -- how many duplicate names exist?
2. Add `NULL_COUNT` on `STG_GOV_PORTAL.NATIONAL_ID` -- are any Gov Portal records missing IDs?
3. Create an expectation: `EXPECT_ERP_HAS_DATA` on ROW_COUNT where `VALUE > 0`

---

**Next:** Open `2_SILVER_LAYER_DQ` to validate transformation logic with custom DMFs.